In [0]:
from pyspark.sql import functions as F

GOLD_BASE = "abfss://gold@skyprint74815.dfs.core.windows.net/skyprint"

fact_source = spark.read.option("header", True).option("inferSchema", True).csv(f"{GOLD_BASE}/final.csv")

fact_source.printSchema()
print("Rows:", fact_source.count())

root
 |-- batch_id: string (nullable = true)
 |-- plane_id: string (nullable = true)
 |-- flight_id: string (nullable = true)
 |-- origin_country: string (nullable = true)
 |-- time_position: timestamp (nullable = true)
 |-- last_contact: timestamp (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- flight_altitude_m: double (nullable = true)
 |-- on_ground: boolean (nullable = true)
 |-- velocity: double (nullable = true)
 |-- true_track: double (nullable = true)
 |-- vertical_rate: double (nullable = true)
 |-- inside_saudi_boundary: boolean (nullable = true)
 |-- registration: string (nullable = true)
 |-- manufacturername: string (nullable = true)
 |-- model: string (nullable = true)
 |-- typecode: string (nullable = true)
 |-- openap_typecode: string (nullable = true)
 |-- aircraft_metadata_matched: boolean (nullable = true)
 |-- weather_batch_available: boolean (nullable = true)
 |-- weather_time: timestamp (nullable = true)
 |-

In [0]:
dim_date = (
    fact_source
    .select(F.to_timestamp("time_position").alias("ts"))
    .distinct()
    .withColumn("date_key", F.date_format("ts", "yyyyMMddHH"))
    .withColumn("full_date", F.to_date("ts"))
    .withColumn("year", F.year("ts"))
    .withColumn("month", F.month("ts"))
    .withColumn("day", F.dayofmonth("ts"))
    .withColumn("hour", F.hour("ts"))
    .withColumn("day_of_week", F.date_format("ts", "EEEE"))
    .select("date_key", "full_date", "year", "month", "day", "hour", "day_of_week")
    .dropDuplicates(["date_key"])
)
dim_date.show(5)

+----------+----------+----+-----+---+----+-----------+
|  date_key| full_date|year|month|day|hour|day_of_week|
+----------+----------+----+-----+---+----+-----------+
|2026092308|2026-09-23|2026|    9| 23|   8|  Wednesday|
|2026092312|2026-09-23|2026|    9| 23|  12|  Wednesday|
|2026092309|2026-09-23|2026|    9| 23|   9|  Wednesday|
|2026092310|2026-09-23|2026|    9| 23|  10|  Wednesday|
|2026092304|2026-09-23|2026|    9| 23|   4|  Wednesday|
+----------+----------+----+-----+---+----+-----------+
only showing top 5 rows


In [0]:
dim_aircraft = (
    fact_source
    .select("plane_id", "registration", "manufacturername", "model", "typecode", "openap_typecode")
    .dropDuplicates(["plane_id"])
)
dim_aircraft.show(5)

+--------+------------+----------------+----------+--------+---------------+
|plane_id|registration|manufacturername|     model|typecode|openap_typecode|
+--------+------------+----------------+----------+--------+---------------+
|  06a2da|      A7-RRR|            NULL|      NULL|    DA40|           NULL|
|  ae5d2f|        NULL|            NULL|      NULL|    NULL|           NULL|
|  8940cb|        NULL|            NULL|      NULL|    NULL|           NULL|
|  801665|        NULL|            NULL|      NULL|    NULL|           NULL|
|  89643f|      A6-AEI|          Airbus|A321 231SL|    A321|           A321|
+--------+------------+----------------+----------+--------+---------------+
only showing top 5 rows


In [0]:
dim_weather_location = (
    fact_source
    .filter(F.col("weather_location_id").isNotNull())
    .select("weather_location_id", "weather_latitude", "weather_longitude")
    .dropDuplicates(["weather_location_id"])
)
dim_weather_location.show(5)
print("Total rows:", dim_weather_location.count())

+-------------------+----------------+-----------------+
|weather_location_id|weather_latitude|weather_longitude|
+-------------------+----------------+-----------------+
|                 57|       25.202108|         51.51188|
|                 32|        24.49912|        54.615387|
|                108|       23.022846|        51.630096|
|                 61|        25.06151|         51.59483|
|                 66|         30.3125|           36.375|
+-------------------+----------------+-----------------+
only showing top 5 rows
Total rows: 165


In [0]:
fact_flight_environmental = (
    fact_source
    .withColumn("flight_obs_id", F.monotonically_increasing_id())
    .withColumn("time_position", F.to_timestamp("time_position"))
    .withColumn("date_key", F.date_format("time_position", "yyyyMMddHH"))
    .select(
        "flight_obs_id", "flight_id", "plane_id", "date_key", "time_position", "batch_id",
        "weather_location_id", "origin_country", "longitude", "latitude", "true_track",
        "flight_altitude_m", "velocity", "derived_airspeed_ms", "vertical_rate", "on_ground",
        "inside_saudi_boundary", "flight_temperature_c", "flight_wind_speed_kmh",
        "flight_wind_direction_deg", "headwind_component_ms", "crosswind_component_ms",
        "estimated_mass_kg", "fuel_flow_kg_s", "co2_g_s", "nox_g_s", "co_g_s",
        "hc_g_s", "sox_g_s", "soot_g_s", "weather_distance_km", "pressure_level_hpa",
        "aircraft_metadata_matched", "airspeed_plausible", "fuel_flow_ready", "fuel_flow_success"
    )
)
fact_flight_environmental.show(5)

+-------------+---------+--------+----------+-------------------+---------------+-------------------+--------------+---------+--------+----------+-----------------+--------+-------------------+-------------+---------+---------------------+--------------------+---------------------+-------------------------+---------------------+----------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------------+------------------+---------------+-----------------+
|flight_obs_id|flight_id|plane_id|  date_key|      time_position|       batch_id|weather_location_id|origin_country|longitude|latitude|true_track|flight_altitude_m|velocity|derived_airspeed_ms|vertical_rate|on_ground|inside_saudi_boundary|flight_temperature_c|flight_wind_speed_kmh|flight_wind_direction_deg|headwind_component_ms|crosswind_component_ms|estimated_mass_kg|    f

In [0]:
in_dims = set(dim_aircraft.columns + dim_date.columns + dim_weather_location.columns)
missing = [c for c in fact_source.columns
           if c not in fact_flight_environmental.columns and c not in in_dims]
print(len(missing), "columns not in the star schema:")
print(missing)

3 columns not in the star schema:
['last_contact', 'weather_batch_available', 'weather_time']


In [0]:
tables = {
    "fact_flight_environmental": fact_flight_environmental,
    "dim_aircraft": dim_aircraft,
    "dim_date": dim_date,
    "dim_weather_location": dim_weather_location,
}

for name, df in tables.items():
    path = f"{GOLD_BASE}/dimensional_model/{name}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"skyprint_databricks.default.{name}")
    print(f"Saved {name}: {df.count()} rows -> {path}")

Saved fact_flight_environmental: 62340 rows -> abfss://gold@skyprint74815.dfs.core.windows.net/skyprint/dimensional_model/fact_flight_environmental
Saved dim_aircraft: 3355 rows -> abfss://gold@skyprint74815.dfs.core.windows.net/skyprint/dimensional_model/dim_aircraft
Saved dim_date: 122 rows -> abfss://gold@skyprint74815.dfs.core.windows.net/skyprint/dimensional_model/dim_date
Saved dim_weather_location: 165 rows -> abfss://gold@skyprint74815.dfs.core.windows.net/skyprint/dimensional_model/dim_weather_location
